# 🏠 House Price Prediction Portfolio Project

Welcome to the first portfolio project for **Linear Regression**. In this notebook, we will predict house prices based on various property features.

## 1. Business Problem
A real estate agency wants to automate the valuation of properties. They want a machine learning model that takes in property characteristics (size, location, age) and outputs an estimated market price. This will help their agents set competitive listing prices and identify undervalued properties for investors.

## 2. Dataset Description
We will generate a synthetic dataset representing typical real estate data.

**Features:**
- `SquareFeet`: Size of the house.
- `NumBedrooms`: Number of bedrooms.
- `HouseAge`: Age of the house in years.
- `DistanceToCityCenter`: Distance in miles.
- **Target:** `Price` (Continuous variable).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
n_samples = 2000

sqft = np.random.normal(2000, 500, n_samples)
bedrooms = np.random.randint(1, 6, n_samples)
age = np.random.randint(0, 100, n_samples)
distance = np.random.exponential(10, n_samples)

# Base price + feature contributions + noise
price = 50000 + 150 * sqft + 10000 * bedrooms - 500 * age - 2000 * distance + np.random.normal(0, 30000, n_samples)

df = pd.DataFrame({
    'SquareFeet': sqft,
    'NumBedrooms': bedrooms,
    'HouseAge': age,
    'DistanceToCityCenter': distance,
    'Price': price
})

df.head()

## 3. Exploratory Data Analysis (EDA)
Let's check the correlation of features with Price.

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.show()

sns.scatterplot(x='SquareFeet', y='Price', data=df, alpha=0.3)
plt.title('Square Feet vs Price')
plt.show()

## 4. Feature Engineering
For Linear Regression, scaling isn't strictly required if we have a closed-form solution (Normal Equation), but it's highly recommended for gradient descent and regularized versions.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop('Price', axis=1)
y = df['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 5. Model Training & 6. Hyperparameter Tuning
We will use Ridge Regression (Linear Regression with L2 penalty) and tune the alpha parameter.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

param_grid = {'alpha': [0.1, 1.0, 10.0, 100.0]}
grid = GridSearchCV(Ridge(), param_grid, cv=5, scoring='neg_mean_absolute_error')
grid.fit(X_train_scaled, y_train)

best_model = grid.best_estimator_
print("Best Alpha:", grid.best_params_['alpha'])

## 7. Evaluation & 8. Error Analysis

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = best_model.predict(X_test_scaled)

print("MAE: $", round(mean_absolute_error(y_test, y_pred), 2))
print("RMSE: $", round(np.sqrt(mean_squared_error(y_test, y_pred)), 2))
print("R^2 Score:", round(r2_score(y_test, y_pred), 4))

# Error Analysis: Plotting Residuals
residuals = y_test - y_pred
sns.histplot(residuals, kde=True)
plt.title('Distribution of Residuals')
plt.xlabel('Prediction Error ($)')
plt.show()

## 9. Visualizations & 10. Business Insights

In [ ]:
importance = pd.DataFrame({'Feature': X.columns, 'Coefficient': best_model.coef_})
importance = importance.sort_values('Coefficient', ascending=False)

sns.barplot(x='Coefficient', y='Feature', data=importance, orient='h')
plt.title('Impact of Features on House Price')
plt.show()

print("Business Insights:")
print("- **Square Footage** is overwhelmingly the strongest driver of price.")
print("- **Distance from City Center** heavily penalizes the price. Suburban homes are cheaper.")

## 11. Future Improvements
- Capture non-linear relationships (e.g., adding `Age^2`) or use Polynomial Regression.